# Epistemic states through the Jacobian lens — full battery

Reproduces the results of *Verdict Collapse and Sustained Indeterminacy:
Reading Epistemic States of Language Models with the Jacobian Lens*.

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Total time ≈ 25 min.

In [ ]:
!git clone --depth 1 https://github.com/anthropics/jacobian-lens.git
!git clone --depth 1 https://github.com/mleyvaz/jspace-epistemic-lens.git
%cd jacobian-lens
!pip install -q -e .
!nvidia-smi | head -12

In [ ]:
import torch, transformers, jlens
jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
hf_model = transformers.AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).cuda()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)

lens = jlens.JacobianLens.from_pretrained(
    "neuronpedia/jacobian-lens",
    filename="qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    revision="qwen-n1000",
)
print("layers:", model.n_layers)

In [ ]:
import json, sys
sys.path.append("/content/jspace-epistemic-lens/src")
import jspace

lexids = jspace.build_lexicons(tokenizer)
print({k: len(v) for k, v in lexids.items()}, "token ids per lexicon")

with open("/content/jspace-epistemic-lens/stimuli/stimuli.json") as f:
    STIMULI = json.load(f)
print({k: len(v) for k, v in STIMULI.items()}, "stimuli per condition")

In [ ]:
import pandas as pd

LAYERS = list(range(18, 31))
rows, done = [], 0
total = sum(len(v) for v in STIMULI.values())
for cond, prompts in STIMULI.items():
    for i, p in enumerate(prompts):
        jl, ml, _ = lens.apply(model, p, layers=LAYERS, positions=[-2])
        for L in LAYERS:
            m, ent = jspace.log_masses(jl[L][0], lexids)
            rows.append(dict(cond=cond, item=i, layer=L, ent=ent,
                             log_T=m["T"], log_I=m["I"], log_F=m["F"]))
        done += 1
        if done % 10 == 0:
            print(f"{done}/{total} prompts")

df = pd.DataFrame(rows)
df.to_csv("jspace_battery.csv", index=False)
print("saved jspace_battery.csv:", df.shape)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(7)
def boot_ci(a, n=4000):
    a = np.asarray(a)
    means = a[rng.integers(0, len(a), (n, len(a)))].mean(axis=1)
    return a.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

colors = {"T": "tab:blue", "I": "tab:orange", "F": "tab:green"}
titles = {"conflict": "Conflicting evidence", "kable": "First-person false belief",
          "control": "Factual control"}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, cond in zip(axes, ["conflict", "kable", "control"]):
    sub = df[df.cond == cond]
    for comp, c in colors.items():
        mu, lo, hi = [], [], []
        for L in LAYERS:
            m_, l_, h_ = boot_ci(sub[sub.layer == L][f"log_{comp}"].values)
            mu.append(m_); lo.append(l_); hi.append(h_)
        ax.plot(LAYERS, mu, marker="o", ms=3.5, color=c, label=comp)
        ax.fill_between(LAYERS, lo, hi, color=c, alpha=0.18)
    ax.axvspan(17.5, 21.5, color="gray", alpha=0.13)
    ax.set_title(f"{titles[cond]} (n={sub.item.nunique()})", fontsize=10)
    ax.set_xlabel("layer"); ax.grid(alpha=0.3)
axes[0].set_ylabel("log10 lexicon probability mass")
axes[0].legend(title="lexicon")
plt.tight_layout(); plt.savefig("fig1_logmass.png", dpi=220); plt.show()

from google.colab import files
files.download("jspace_battery.csv")
files.download("fig1_logmass.png")

Full statistics (per-layer contrasts, collapse magnitudes, coexistence counts,
robustness): run `analysis/statistics.py`, `analysis/figures.py`, and
`analysis/robustness.py` from the repository root on the CSV produced above.

License: this repo MIT; `jlens` code and lenses are Anthropic's (Apache 2.0),
downloaded at run time; model weights under their own licenses.